In [ ]:
# Fetching MTD and Historical Data from SQL

import pandas as pd
import pyodbc
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",        # MySQL Workbench ka Hostname
    user="root",             # MySQL username
    password="12345",
    database="data",           # Default Schema
    port=3306
)

query = """SELECT * FROM cy_data WHERE TRANSACTION_DATE >= DATE_FORMAT(CURDATE(), '%Y-%m-01');"""
query1 = """SELECT * FROM cy_data WHERE TRANSACTION_DATE < DATE_FORMAT(CURDATE(), '%Y-%m-01');"""

MTD = pd.read_sql(query, conn)
CY_1 = pd.read_sql(query1, conn)


In [ ]:
# Product Master
ND=pd.read_excel(r"D:\Lovepreet\Master File\Master.xlsb", sheet_name="New-Depot")
PM=pd.read_excel(r"D:\Lovepreet\Master File\Master.xlsb", sheet_name="New- Product Master")
PD=pd.read_excel(r"D:\Lovepreet\Master File\Master.xlsb", sheet_name="Product")

In [ ]:
## Data Cleaning

import pandas as pd
import numpy as np
from datetime import date

MTD=MTD[["HISTORY_ID","DEPOT_ID","STATE_NAME","CARD_NO","NAME","MOBILE",
    "DEALER_NAME","DEALER_CODE","SKU_CODE","DESCRIPTION","PRODUCT_NAME",
    "VOLUME","QUANTITY","TOTAL_POINTS","POINTS_VIA","TRANSACTION_DATE",
    "CLASSIFICATION_CODE","PAINTER_STATUS","TOKEN_CODE","PAINTER_ID"]]


cond = MTD["DEPOT_ID"].isna() | (MTD["DEPOT_ID"] == "")
MTD.loc[:, "Final_depot_id"] = np.where(
    cond,
    np.where(
        MTD["STATE_NAME"].notna() & (MTD["STATE_NAME"] != ""),
        MTD["STATE_NAME"],
        "Others"
    ),
    MTD["DEPOT_ID"]
)

MTD["TRANSACTION_DATE"] = pd.to_datetime(MTD["TRANSACTION_DATE"],format="mixed",dayfirst=True,errors="coerce")
MTD.drop(MTD[(MTD["POINTS_VIA"] == "REFUND_POINTS") | (MTD["TRANSACTION_DATE"].dt.date == date.today())].index,inplace=True)

df = MTD.copy()

df["Volume_in_Ltr"] = df["VOLUME"] * df["QUANTITY"]

df = df.merge(
    PM[["Product Name In Pragati", "New Rate"]],
    left_on="PRODUCT_NAME",
    right_on="Product Name In Pragati",
    how="left")

df["ASP"] = df["New Rate"]

df["Value"] = df["Volume_in_Ltr"] * df["ASP"]

df = df.merge(
    ND[["Proper(Depot_ID)", "Depot Code","ZONE","Division","Depot Name Updated"]],
    left_on="Final_depot_id",
    right_on="Proper(Depot_ID)",
    how="left")

df = df.merge(
    PD[["DESCRIPTION", "PRODUCT_NAME", "Category"]],
    on="DESCRIPTION",
    how="left")

df.drop(["DEPOT_ID", "PRODUCT_NAME_x"], axis=1, inplace=True)

df.rename(
    columns={
        "PRODUCT_NAME_y": "PRODUCT_NAME",
        "Category": "CATEGORY_CODE",
        "DEPOT_ID_y": "DEPOT_ID",
        "Proper(Depot_ID)" : "Depot_ID"
    },
    inplace=True)

Final = df[
    [
        "HISTORY_ID","Final_depot_id","STATE_NAME","CARD_NO","NAME","MOBILE","DEALER_NAME",
        "DEALER_CODE","SKU_CODE","DESCRIPTION","PRODUCT_NAME",
        "VOLUME","QUANTITY","TOTAL_POINTS","POINTS_VIA","TRANSACTION_DATE",
        "CATEGORY_CODE","CLASSIFICATION_CODE","PAINTER_STATUS",
        "TOKEN_CODE","PAINTER_ID","Volume_in_Ltr","ASP",
        "Value","Depot Code","ZONE","Division","Depot Name Updated"
    ]
].copy()

Final.rename(columns={"Final_depot_id": "DEPOT_ID"}, inplace=True)

Final['Quarter'] = np.where(
    Final['TRANSACTION_DATE'].notna(),
    'Q' + (((Final['TRANSACTION_DATE'].dt.month - 4) % 12 // 3) + 1).astype(str),
    'NA'
)

Final.loc[(Final["Division"]=="") | (Final["Division"].isna()),["Division"]]="NA"
Final.loc[(Final["DEPOT_ID"]=="Others"),["ZONE", "Division", "Depot Code", "Depot Name Updated"]] = "N/A"


Final.to_csv("Final_MTD.csv",index=False)

Unique1=Final[["MOBILE","DEPOT_ID","Depot Code","ZONE","Division",
       "Depot Name Updated","TRANSACTION_DATE"]
     ].sort_values("TRANSACTION_DATE",ascending=True).drop_duplicates(subset="MOBILE",keep="last")

Unique=Unique1[["MOBILE","DEPOT_ID","Depot Code","ZONE","Division","Depot Name Updated"]]

CY_1["MOBILE"] = CY_1["MOBILE"].astype(str)
Unique["MOBILE"] = Unique["MOBILE"].astype(str)

CY_1 = CY_1.merge(Unique, on="MOBILE", how="left", suffixes=("", "_u"))

cols = ["DEPOT_ID","Depot Code","ZONE","Division","Depot Name Updated"]

for c in cols:
    if c + "_u" in CY_1.columns:
        CY_1[c] = CY_1[c + "_u"]

CY_1 = CY_1.drop(columns=['DEPOT_ID_u', 'Depot Code_u', 'ZONE_u', 'Division_u', 'Depot Name Updated_u'])

CY_Data = pd.concat([CY_1, Final], ignore_index=True)
CY_Data["TRANSACTION_DATE"] = pd.to_datetime(CY_Data["TRANSACTION_DATE"],format="mixed",dayfirst=True,errors="coerce")
CY_Data["DEALER_CODE"] = (pd.to_numeric(CY_Data["DEALER_CODE"], errors="coerce").fillna(0).astype(int).astype(str))
CY_Data["MOBILE"]=CY_Data["Mobile"].astype(int)
CY_Data["MOBILE"] = pd.to_numeric(CY_Data["MOBILE"], errors="coerce").fillna(0).astype(int)
CY_Data.to_csv("CY_Data.csv",index=False)

In [ ]:
# OOTY TOY TRAIN SCHEME

Data = CY_Data[
    (CY_Data["VOLUME"].isin([10, 20])) &
    (CY_Data["DEALER_CODE"].isin(['165465','166546','166547','168558','182184','193288','198228','210422'])) &
    (CY_Data["TRANSACTION_DATE"].between("2025-08-01", "2026-01-31")) &
    (CY_Data["PRODUCT_NAME"].isin([
        'EXCEL ANTIPEEL','EXCEL NO DUST WHITE','EXCEL ANTIPEEL WOW WHITE','EXCEL MICA MARBLE',
        'EXCEL MM S&S','EXCEL TOTAL','EXCEL TOPGUARD','EXCEL EVERLAST','BEAUTY GOLD',
        'BEAUTY GOLD WASHABLE','IMP HD NXT','IMP KASH MATT','IMP KASHMIR','IMP SHEEN',
        'IMPRESSIONS GLITTER','IMPRESSIONS HD','IMPRESSIONS UHD'
    ]))
]

Pivot=Data.pivot_table(index="DEALER_CODE",values="Volume_in_Ltr",aggfunc="sum")
with pd.ExcelWriter("OOTY TOY TRAIN SCHEME.xlsx") as W:
    Data.to_excel(W,sheet_name="Raw",index=False)
    Pivot.to_excel(W,sheet_name="Pivot",index=False)


In [ ]:
# SPECIAL PAINTER SCHEME 

Data = CY[
    (CY["DEPOT_ID"].isin(["D813", "S813","D820","S820"])) &
    (CY["TRANSACTION_DATE"].between("2025-09-21", "2025-11-21")) &
    (CY["PRODUCT_NAME"].isin([
        'IMP KASH MATT', 'IMP KASHMIR', 'IMP SHEEN', 'IMPRESSIONS GLITTER',
        'IMPRESSIONS HD', 'IMPRESSIONS UHD', 'IMP HD NXT', 'EXCEL ANTIPEEL',
        'EXCEL ANTIPEEL WOW WHITE', 'EXCEL EVERLAST', 'EXCEL EXTERIOR TEXTURE RIGOR',
        'EXCEL MICA MARBLE', 'EXCEL MM S&S', 'EXCEL MM S&S NXT', 'EXCEL NO DUST WHITE',
        'EXCEL NXT', 'EXCEL TOPGUARD', 'EXCEL TOTAL', 'EXCEL TOTAL FLOOR COAT',
        'EXCEL VIRUS GUARD', 'EXCEL SHEEN', 'BEAUTY GOLD WASHABLE', 'BEAUTY GOLD'
    ]))
]
Pivot=Data.pivot_table(index="MOBILE",values="Volume_in_Ltr",aggfunc="sum")
with pd.ExcelWriter("DIWALI SPECIAL PAINTER SCHEME GUJARAT.xlsx") as writer:
    Data.to_excel(writer,sheet_name="Raw")
    Pivot.to_excel(writer,sheet_name="Pivot")